In [15]:
import os
from pathlib import Path
import sys


# 프로젝트 루트 경로를 찾아 src 폴더를 sys.path에 추가합니다.
def find_project_root() -> Path:
    curr = Path.cwd()
    for parent in [curr] + list(curr.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    return curr

PROJECT_ROOT = find_project_root()
src_dir = str(PROJECT_ROOT / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

os.environ.setdefault("LANGSMITH_TRACING", "false")

'true'

In [18]:
import pandas as pd


# 1) 데이터 불러오기
members_file_path = str(PROJECT_ROOT / "data/raw/csv/members_v1.csv")
consumption_file_path = str(PROJECT_ROOT / "data/raw/csv/consumption_v1.csv")


members = pd.read_csv(members_file_path)
consumption = pd.read_csv(consumption_file_path)


In [19]:
# 2) 컬럼명 확인
print("members columns:", members.columns.tolist())
print("consumption columns:", consumption.columns.tolist())

members columns: ['id', 'name', 'age', '직업', '성별', '연봉', '지역', '최상위 카드등급', '페르소나']
consumption columns: ['멤버 id', 'id', '사용 금액', '사용 시간', '결제 내역', '결제 장소 (가맹점 여부)', '할부 여부', '할부 개월', '할부 무/유이자 여부', '거래 상태 (승인 / 취소)', '해외 결제', '업종 카테고리', '결제 방식 (온/오프라인)']


In [3]:
# 3) 날짜/숫자형 변환
consumption["사용 시간"] = pd.to_datetime(consumption["사용 시간"])
consumption["사용 금액"] = pd.to_numeric(consumption["사용 금액"], errors="coerce")

In [25]:
# 4) 예시: 특정 유저(id=1) 선택
def analyze_user(user_id, members, consumption):

    user_info = members[members["id"] == user_id]
    user_consumption = consumption[consumption["멤버 id"] == user_id].copy()

    if user_consumption.empty:
        return None

    total_spending = user_consumption["사용 금액"].sum()

    return {
        "user_id": user_id,
        "user_info": user_info,
        "user_consumption": user_consumption,
        "total_spending": total_spending
    }

result = analyze_user(1, members, consumption)
print("\n[유저 정보]")
print(result["user_info"])

print("\n[유저 소비 데이터 상위 5개]")
print(result["user_consumption"].head())

print("\n[총 소비금액]")
print(result["total_spending"])


[유저 정보]
   id name  age    직업 성별    연봉  지역 최상위 카드등급  페르소나
0   1  김배달   32  프리랜서  남  3500  서울     Gold  배달중독

[유저 소비 데이터 상위 5개]
   멤버 id     id  사용 금액             사용 시간  결제 내역 결제 장소 (가맹점 여부) 할부 여부  할부 개월  \
0      1  20001   9009  2024-01-01 08:57   스타벅스              Y     N      0   
1      1  20002  13952  2024-01-02 12:25   일반식당              Y     N      0   
2      1  20003  14986  2024-01-03 13:10   일반식당              Y     N      0   
3      1  20004  11861  2024-01-04 12:23   일반식당              Y     N      0   
4      1  20005  40115  2024-01-04 18:56  배달의민족              Y     N      0   

  할부 무/유이자 여부 거래 상태 (승인 / 취소) 해외 결제 업종 카테고리 결제 방식 (온/오프라인)  
0           -              승인     N      식비    오프라인 - 실물카드  
1           -              승인     N      식비    오프라인 - 삼성페이  
2           -              승인     N      식비    오프라인 - 실물카드  
3           -              승인     N      식비    오프라인 - 삼성페이  
4           -              승인     N      식비    온라인 - 카카오페이  

[총 소비금액]
1746539


In [27]:
# 5) 기본 확인
print("\n[유저 정보]")
print(result["user_info"])

print("\n[유저 소비 데이터 상위 5개]")
print(result["user_consumption"].head())


[유저 정보]
   id name  age    직업 성별    연봉  지역 최상위 카드등급  페르소나
0   1  김배달   32  프리랜서  남  3500  서울     Gold  배달중독

[유저 소비 데이터 상위 5개]
   멤버 id     id  사용 금액             사용 시간  결제 내역 결제 장소 (가맹점 여부) 할부 여부  할부 개월  \
0      1  20001   9009  2024-01-01 08:57   스타벅스              Y     N      0   
1      1  20002  13952  2024-01-02 12:25   일반식당              Y     N      0   
2      1  20003  14986  2024-01-03 13:10   일반식당              Y     N      0   
3      1  20004  11861  2024-01-04 12:23   일반식당              Y     N      0   
4      1  20005  40115  2024-01-04 18:56  배달의민족              Y     N      0   

  할부 무/유이자 여부 거래 상태 (승인 / 취소) 해외 결제 업종 카테고리 결제 방식 (온/오프라인)  
0           -              승인     N      식비    오프라인 - 실물카드  
1           -              승인     N      식비    오프라인 - 삼성페이  
2           -              승인     N      식비    오프라인 - 실물카드  
3           -              승인     N      식비    오프라인 - 삼성페이  
4           -              승인     N      식비    온라인 - 카카오페이  


In [29]:
# 6) 온보딩용 요약 지표 만들기
total_spending = result["user_consumption"]["사용 금액"].sum()
transaction_count = len(result["user_consumption"])

# 업종별 소비 합계
category_sum = (
    result["user_consumption"].groupby("업종 카테고리")["사용 금액"]
    .sum()
    .sort_values(ascending=False)
)

# 결제 내역별 상위 소비처
merchant_sum = (
    result["user_consumption"].groupby("결제 내역")["사용 금액"]
    .sum()
    .sort_values(ascending=False)
)

# 야간 소비 (예: 21시 이후)
result["user_consumption"]["사용 시간"] = pd.to_datetime(
    result["user_consumption"]["사용 시간"],
    errors="coerce"
)

result["user_consumption"]["hour"] = result["user_consumption"]["사용 시간"].dt.hour
night_spending = result["user_consumption"][result["user_consumption"]["hour"] >= 21]["사용 금액"].sum()
night_ratio = night_spending / total_spending if total_spending > 0 else 0

# 월별 소비
result["user_consumption"]["year_month"] = result["user_consumption"]["사용 시간"].dt.to_period("M")
monthly_spending = (
    result["user_consumption"].groupby("year_month")["사용 금액"]
    .sum()
    .sort_index()
)

print("\n[총 소비 금액]")
print(total_spending)

print("\n[총 거래 수]")
print(transaction_count)

print("\n[업종별 소비 합계]")
print(category_sum)

print("\n[상위 소비처]")
print(merchant_sum.head(5))

print("\n[야간 소비 비율]")
print(round(night_ratio, 2))

print("\n[월별 소비]")
print(monthly_spending)


[총 소비 금액]
1746539

[총 거래 수]
95

[업종별 소비 합계]
업종 카테고리
식비    1746539
Name: 사용 금액, dtype: int64

[상위 소비처]
결제 내역
배달의민족     497080
쿠팡이츠      465673
일반식당      212361
투썸플레이스    196832
스타벅스      195901
Name: 사용 금액, dtype: int64

[야간 소비 비율]
0.27

[월별 소비]
year_month
2024-01    555708
2024-02    648707
2024-03    542124
Freq: M, Name: 사용 금액, dtype: int64


In [31]:
# 7) 유저 기본 정보 추출
if not result["user_info"].empty:
    user_name = result["user_info"].iloc[0]["name"]
    age = result["user_info"].iloc[0]["age"]
    job = result["user_info"].iloc[0]["직업"]
    persona = result["user_info"].iloc[0]["페르소나"]
    region = result["user_info"].iloc[0]["지역"]
else:
    user_name = "알 수 없음"
    age = "알 수 없음"
    job = "알 수 없음"
    persona = "알 수 없음"
    region = "알 수 없음"


top_category = category_sum.index[0] if not category_sum.empty else "없음"
top_merchant = merchant_sum.index[0] if not merchant_sum.empty else "없음"

user_profile_text = f"""
이 사용자의 이름은 {user_name}이고, 나이는 {age}세, 직업은 {job}, 지역은 {region}입니다.
현재 페르소나는 {persona}입니다.
최근 3개월 총 소비금액은 {int(total_spending):,}원이며, 총 거래 수는 {transaction_count}건입니다.
가장 많이 소비한 업종 카테고리는 {top_category}입니다.
가장 많이 사용한 결제처는 {top_merchant}입니다.
야간 소비 비율은 {round(night_ratio * 100, 1)}%입니다.
"""

print("\n[유저 프로필 요약]")
print(user_profile_text)


[유저 프로필 요약]

이 사용자의 이름은 김배달이고, 나이는 32세, 직업은 프리랜서, 지역은 서울입니다.
현재 페르소나는 배달중독입니다.
최근 3개월 총 소비금액은 1,746,539원이며, 총 거래 수는 95건입니다.
가장 많이 소비한 업종 카테고리는 식비입니다.
가장 많이 사용한 결제처는 배달의민족입니다.
야간 소비 비율은 26.5%입니다.



In [32]:
user_memory = {
    "user_id": result["user_id"],
    "user_name": user_name,
    "persona": persona,
    "total_spending": int(total_spending),
    "top_category": top_category,
    "top_merchant": top_merchant,
    "night_ratio": round(night_ratio, 3),
    "last_feedback": "",
    "score": 0
}

print("\n[초기 메모리]")
print(user_memory)


[초기 메모리]
{'user_id': 1, 'user_name': '김배달', 'persona': '배달중독', 'total_spending': 1746539, 'top_category': '식비', 'top_merchant': '배달의민족', 'night_ratio': np.float64(0.265), 'last_feedback': '', 'score': 0}
